# Tutorial: 42-bin mean-field calculations

Change the parameters in each panel and run the notebook from top to bottom. All calculations are performed by `src/mean_field_42bin.py`.

In [ ]:
from pathlib import Path
import sys
import numpy as np

repo = Path.cwd().resolve()
if repo.name == 'notebooks':
    repo = repo.parent
sys.path.insert(0, str(repo / 'src'))

import mean_field_42bin as mf
np.set_printoptions(precision=6, suppress=True, linewidth=180)

## 1. Fixed point and stability

The analytic Jacobian is evaluated at the fixed point. The fixed point is stable when the modulus of every eigenvalue is below one.

In [ ]:
# Parameters
alpha2 = 0.03
fs = 0.50

fixed = mf.solve_fixed_point_42(alpha2=alpha2, fs=fs)

print('fixed point x* =')
print(fixed['state'])
print('\nJacobian at x* =')
print(fixed['jacobian'])
print(f"\nresidual = {fixed['residual']:.3e}")
print(f"largest |eigenvalue| = {fixed['largest_multiplier']:.10f}")
print('stable =', fixed['largest_multiplier'] < 1.0)

## 2. Bifurcation point

The solver finds a fixed point with one multiplier equal to one. `alpha_guess` is only the starting value for the search.

In [ ]:
# Parameters
fs = 0.50
alpha_guess = 0.21125

critical = mf.solve_bifurcation_42(fs=fs, alpha_guess=alpha_guess)

print(f"alpha2* = {critical['alpha2']:.12f}")
print(f"fs = {critical['fs']:.6f}")
print(f"residual = {critical['residual']:.3e}")
print(f"critical multiplier = {critical['critical_multiplier'].real:.12f}")

## 3. Bifurcation type

The normal-form coefficients classify the point found above without assuming its type beforehand.

In [ ]:
coefficients = mf.normal_form_42(critical)

for name in ('c0', 'c1', 'c_mu', 'c2', 'c_amu', 'c3', 'c5'):
    print(f"{name:5s} = {coefficients[name]:+.8e}")
print('classification =', coefficients['classification'])

## 4. Tricritical point in Fig. 5(a)

The cubic coefficient is checked on both sides of the proposed TP. Its sign change identifies the transition between supercritical and subcritical pitchforks.

In [ ]:
# (label, fs, alpha2 starting value)
test_points = [
    ('left',  0.7956500000, 0.1933500000),
    ('TP',    0.7957070426, 0.1932732499),
    ('right', 0.7958000000, 0.1931400000),
]

tp = mf.verify_tricritical_42(test_points)

for row in tp['rows']:
    point = row['point']
    coefficient = row['coefficients']
    print(f"{row['label']:5s}: fs={point['fs']:.10f}, "
          f"alpha2*={point['alpha2']:.10f}, "
          f"c3={coefficient['c3']:+.6e}, c5={coefficient['c5']:+.6e}")

print('c3 changes sign =', tp['c3_changes_sign'])
print('classification =', 'tricritical pitchfork (TP)' if tp['is_tp'] else 'TP conditions not resolved')